# Modelo de clasificación de emociones — RAVDESS (habla)

Basado en el notebook de referencia **Speech Emotion Recognition 90%** (gemmin, Kaggle):
ZCR + RMS + MFCC cuadro a cuadro, aumentación (ruido, pitch) y una red Conv1D de 5 bloques.

**Diferencias deliberadas respecto a la referencia**

1. *Solo RAVDESS.* La referencia combina RAVDESS, CREMA-D, TESS y SAVEE (~12k clips); aquí hay 1440.
2. *Sin fuga de datos.* La referencia aumenta **antes** de dividir, de modo que copias ruidosas de un mismo clip
   caen en entrenamiento y prueba. Aquí se divide primero y solo se aumenta el entrenamiento.
3. *Evaluación independiente del hablante.* El protocolo principal reserva actores completos para prueba.
4. *Ancho de la red.* Por entrenar en CPU, los filtros se reducen a 1/4 (128-128-64-64-32 en lugar de 512-512-256-256-128);
   la topología es la misma.

Se reportan dos protocolos: **A** (actores 21–24 en prueba, 19–20 en validación) y **B** (división aleatoria
estratificada por clip 80/10/10, comparable con la referencia pero sin fuga).

---
### Cómo ejecutarlo en Kaggle
1. *File → Import Notebook* y sube este archivo.
2. *Settings → Accelerator*: **GPU T4 x2** o **P100**.
3. Fuente de datos, elige una:
   - **Supabase**: *Settings → Internet* **On** y en *Add-ons → Secrets* crea `SUPABASE_URL` y `SUPABASE_KEY`.
   - **Input de Kaggle**: *Add Input* → `orvile/ravdess-dataset`. Se detecta solo y no necesita internet.
4. *Run All*. Los resultados quedan en `/kaggle/working/` (pestaña *Output*).

Con GPU se usa el **ancho original de la referencia** (512-512-256-256-128); en CPU, 1/4 de ese ancho.


In [ ]:
import os, sys, json, time, copy, warnings
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import requests
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if KAGGLE else Path.cwd()
OUT = WORK / "resultados_modelo"; OUT.mkdir(exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WIDTH = (512, 512, 256, 256, 128) if DEVICE.type == "cuda" else (128, 128, 64, 64, 32)
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print("Dispositivo:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "", "| ancho:", WIDTH)
mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False})

## 0. Código del proyecto (incrustado de `src/`)

In [ ]:
# --- src/ravdess_data.py ---
"""Acceso a RAVDESS (habla) almacenado en Supabase.

- Metadatos: tabla public.ravdess_clips (lectura pública vía RLS).
- Audio: bucket público 'ravdess-audio', ruta Actor_XX/<archivo>.wav.

Los .wav se descargan bajo demanda a una caché local (por defecto
~/.cache/ravdess_supabase) que puede borrarse en cualquier momento.
"""
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import requests

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ("SUPABASE_URL", "SUPABASE_KEY"):
        os.environ.setdefault(_name, _secrets.get_secret(_name))
except Exception:
    pass  # fuera de Kaggle o sin secretos: se usan variables de entorno

SUPABASE_URL = os.environ.get("SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_KEY")
BUCKET = "ravdess-audio"
TABLE = "ravdess_clips"
CACHE_DIR = Path(os.environ.get("RAVDESS_CACHE", Path("/tmp/ravdess_supabase")))

EMOTIONS = {1: "neutral", 2: "calm", 3: "happy", 4: "sad",
            5: "angry", 6: "fearful", 7: "disgust", 8: "surprised"}
EMOTIONS_ES = {"neutral": "neutral", "calm": "calma", "happy": "alegría", "sad": "tristeza",
               "angry": "enojo", "fearful": "miedo", "disgust": "asco", "surprised": "sorpresa"}
STATEMENTS = {1: "Kids are talking by the door", 2: "Dogs are sitting by the door"}


def parse_filename(filename):
    """03-01-EE-II-SS-RR-AA.wav -> diccionario de metadatos."""
    modality, channel, emo, inten, stmt, rep, actor = (int(p) for p in Path(filename).stem.split("-"))
    return {
        "filename": Path(filename).name,
        "storage_path": f"Actor_{actor:02d}/{Path(filename).name}",
        "actor": actor,
        "gender": "male" if actor % 2 else "female",
        "emotion_code": emo,
        "emotion": EMOTIONS[emo],
        "intensity": "normal" if inten == 1 else "strong",
        "statement": STATEMENTS[stmt],
        "repetition": rep,
    }


def public_url(storage_path):
    return f"{SUPABASE_URL}/storage/v1/object/public/{BUCKET}/{storage_path}"


def load_metadata():
    """Lee toda la tabla de metadatos vía PostgREST (paginado)."""
    headers = {"apikey": SUPABASE_KEY, "Authorization": f"Bearer {SUPABASE_KEY}"}
    rows, start, step = [], 0, 1000
    while True:
        r = requests.get(f"{SUPABASE_URL}/rest/v1/{TABLE}", params={"select": "*", "order": "filename"},
                         headers={**headers, "Range": f"{start}-{start + step - 1}"}, timeout=60)
        r.raise_for_status()
        batch = r.json()
        rows += batch
        if len(batch) < step:
            break
        start += step
    df = pd.DataFrame(rows)
    df["emotion_es"] = df["emotion"].map(EMOTIONS_ES)
    return df


def _fetch(storage_path):
    dest = CACHE_DIR / storage_path
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        for attempt in range(5):
            try:
                r = requests.get(public_url(storage_path), timeout=60)
                r.raise_for_status()
                break
            except requests.RequestException:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)
        tmp = dest.with_suffix(".part")
        tmp.write_bytes(r.content)
        tmp.rename(dest)
    return str(dest)


def ensure_audio(df, workers=8):
    """Descarga (si hace falta) los audios de df y agrega la columna 'path'."""
    with ThreadPoolExecutor(workers) as ex:
        paths = list(ex.map(_fetch, df["storage_path"]))
    return df.assign(path=paths)


In [ ]:
# --- src/ser_model.py ---
"""Pipeline de clasificación de emociones basado en el notebook de referencia
"Speech Emotion Recognition 90%" (gemmin, Kaggle):

- 2.5 s de audio desde el segundo 0.6, a 22 050 Hz.
- Vector por clip = [ZCR | RMS | MFCC(20)] cuadro a cuadro, aplanado (2376 valores).
- Aumentación: original, + ruido, + pitch-shift con ruido (3 versiones por clip).
- Red Conv1D de 5 bloques (Conv-BN-MaxPool) + Dense 512 + softmax.

Diferencia deliberada con la referencia: la aumentación se aplica *después* de
dividir los datos y solo al entrenamiento, y la división es por actor, para que
el conjunto de prueba contenga hablantes nunca vistos.
"""
import warnings

import librosa
import numpy as np
import torch
from torch import nn

SR = 22050
DURATION = 2.5
OFFSET = 0.6
FRAME, HOP = 2048, 512
N_SAMPLES = int(SR * DURATION)


def load_clip(path):
    warnings.filterwarnings("ignore")
    y, _ = librosa.load(path, sr=SR, mono=True, duration=DURATION, offset=OFFSET)
    return librosa.util.fix_length(y, size=N_SAMPLES)


def noise(y, rng, rate=0.035):
    return y + rate * rng.uniform() * np.max(np.abs(y)) * rng.normal(size=y.shape)


def pitch(y, steps=0.7):
    return librosa.effects.pitch_shift(y, sr=SR, n_steps=steps)


def frame_features(y):
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=FRAME, hop_length=HOP)[0]
    rms = librosa.feature.rms(y=y, frame_length=FRAME, hop_length=HOP)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20, n_fft=FRAME, hop_length=HOP).T.ravel()
    return np.concatenate([zcr, rms, mfcc]).astype(np.float32)


def clip_variants(args):
    """(path, augment, seed) -> lista de vectores: original [+ ruido, + pitch con ruido]."""
    path, augment, seed = args
    y = load_clip(path)
    out = [frame_features(y)]
    if augment:
        rng = np.random.default_rng(seed)
        out.append(frame_features(noise(y, rng)))
        out.append(frame_features(noise(pitch(y), rng)))
    return out


class ConvSER(nn.Module):
    """Equivalente en PyTorch de la red Conv1D de la referencia (Keras)."""

    def __init__(self, n_features, n_classes, width=(512, 512, 256, 256, 128)):
        super().__init__()

        def block(cin, cout, k, pool_k, drop):
            layers = [nn.Conv1d(cin, cout, k, padding="same"), nn.ReLU(), nn.BatchNorm1d(cout),
                      nn.MaxPool1d(pool_k, stride=2, padding=pool_k // 2)]
            return layers + ([nn.Dropout(drop)] if drop else [])

        c1, c2, c3, c4, c5 = width
        self.features = nn.Sequential(
            *block(1, c1, 5, 5, 0), *block(c1, c2, 5, 5, 0.2), *block(c2, c3, 5, 5, 0),
            *block(c3, c4, 3, 5, 0.2), *block(c4, c5, 3, 3, 0.2),
        )
        with torch.no_grad():
            flat = self.features(torch.zeros(1, 1, n_features)).numel()
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(flat, 512), nn.ReLU(), nn.BatchNorm1d(512),
                                  nn.Linear(512, n_classes))

    def forward(self, x):
        return self.head(self.features(x.unsqueeze(1)))


## 1. Datos desde Supabase y extracción de características

In [ ]:
ORDER = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
ORDER_ES = [EMOTIONS_ES[e] for e in ORDER]

KAGGLE_INPUT = next(Path("/kaggle/input").rglob("Audio_Speech_Actors_01-24"), None) if Path("/kaggle/input").exists() else None
if KAGGLE_INPUT is not None:
    # Input de Kaggle: metadatos a partir de los nombres de archivo (mismo esquema que la tabla de Supabase)
    files = sorted(KAGGLE_INPUT.glob("Actor_*/*.wav"))
    df = pd.DataFrame([{**parse_filename(f), "path": str(f)} for f in files])
    print("Fuente: input de Kaggle")
else:
    assert SUPABASE_URL and SUPABASE_KEY, "Configura los secretos SUPABASE_URL y SUPABASE_KEY o añade el input orvile/ravdess-dataset"
    df = ensure_audio(load_metadata(), workers=16)
    print("Fuente: Supabase")
assert len(df) == 1440, len(df)
df = df.sort_values("filename").reset_index(drop=True)
df["y"] = df["emotion"].map({e: i for i, e in enumerate(ORDER)})
with ProcessPoolExecutor() as ex:
    variants = list(ex.map(clip_variants, [(p, True, SEED + i) for i, p in enumerate(df["path"])], chunksize=8))
V = np.stack([np.stack(v) for v in variants])          # (1440, 3, 2376): original, ruido, pitch+ruido
print(V.shape)

## 2. Entrenamiento

In [ ]:
def make_split(idx_tr, idx_va, idx_te):
    Xtr = V[idx_tr].reshape(-1, V.shape[-1]); ytr = np.repeat(df["y"].values[idx_tr], 3)   # aumentado
    Xva, yva = V[idx_va, 0], df["y"].values[idx_va]                                          # solo original
    Xte, yte = V[idx_te, 0], df["y"].values[idx_te]
    sc = StandardScaler().fit(Xtr)
    t = lambda a: torch.tensor(sc.transform(a), dtype=torch.float32)
    return (t(Xtr), torch.tensor(ytr)), (t(Xva), torch.tensor(yva)), (t(Xte), torch.tensor(yte))

def predict(model, X, bs=256):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + bs].to(DEVICE)).argmax(1).cpu() for i in range(0, len(X), bs)]).numpy()

def train(tr, va, epochs=50, bs=64, patience=8):
    model = ConvSER(tr[0].shape[1], len(ORDER), width=WIDTH).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3, min_lr=1e-5)
    hist, best, best_state, wait = [], -1, None, 0
    for ep in range(epochs):
        Xg, yg = tr[0].to(DEVICE), tr[1].to(DEVICE)
        model.train(); perm = torch.randperm(len(tr[0]), device=DEVICE); loss_sum = 0; t0 = time.time()
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            if len(b) < 2: continue
            opt.zero_grad(); loss = nn.functional.cross_entropy(model(Xg[b]), yg[b]); loss.backward(); opt.step()
            loss_sum += loss.item() * len(b)
        acc_tr = accuracy_score(tr[1][:2000:3], predict(model, tr[0][:2000:3]))
        acc_va = accuracy_score(va[1], predict(model, va[0]))
        sched.step(acc_va)
        hist.append({"epoca": ep + 1, "loss": loss_sum / len(perm), "acc_train": acc_tr, "acc_val": acc_va, "lr": opt.param_groups[0]["lr"]})
        print(f"ep {ep+1:2d} loss {hist[-1]['loss']:.3f} acc_tr {acc_tr:.3f} acc_val {acc_va:.3f} ({time.time()-t0:.0f}s)")
        if acc_va > best:
            best, best_state, wait = acc_va, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

def evaluate(name, model, te, hist):
    pred = predict(model, te[0]); y = te[1].numpy()
    res = {"protocolo": name, "accuracy": accuracy_score(y, pred), "f1_macro": f1_score(y, pred, average="macro"),
           "n_test": len(y), "epocas": len(hist)}
    print(classification_report(y, pred, target_names=ORDER_ES, digits=3))
    cm = confusion_matrix(y, pred, normalize="true")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), gridspec_kw={"width_ratios": [1, 1.2]})
    axes[0].plot(hist.epoca, hist.acc_train, color="#2a78d6", lw=2, label="entrenamiento")
    axes[0].plot(hist.epoca, hist.acc_val, color="#eb6834", lw=2, label="validación")
    axes[0].set_xlabel("época"); axes[0].set_ylabel("accuracy"); axes[0].legend(frameon=False); axes[0].grid(alpha=.3)
    axes[0].set_title(f"Curvas de aprendizaje — {name}", loc="left", fontweight="bold")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=ORDER_ES, yticklabels=ORDER_ES,
                ax=axes[1], cbar=False, linewidths=1, linecolor="white", annot_kws={"size": 8})
    axes[1].set_xlabel("predicción"); axes[1].set_ylabel("real")
    axes[1].set_title(f"Matriz de confusión (normalizada) — acc {res['accuracy']:.1%}", loc="left", fontweight="bold")
    fig.tight_layout(); fig.savefig(OUT / f"{name.split()[0].lower()}_{name.split()[1]}.png")
    plt.show()
    return res

### Protocolo A — independiente del hablante (actores 21–24 en prueba)

In [ ]:
actor = df["actor"].values
idx_te = np.where(actor >= 21)[0]; idx_va = np.where((actor == 19) | (actor == 20))[0]; idx_tr = np.where(actor <= 18)[0]
tr, va, te = make_split(idx_tr, idx_va, idx_te)
print("train (aumentado)", len(tr[0]), "| val", len(va[0]), "| test", len(te[0]))
model_a, hist_a = train(tr, va)
res_a = evaluate("Protocolo A", model_a, te, hist_a)
torch.save(model_a.state_dict(), MODELS / "convser_protocolo_a.pt")
res_a

### Protocolo B — división aleatoria por clip (80/10/10, estratificada)

In [ ]:
idx = np.arange(len(df))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.2, stratify=df["y"], random_state=SEED)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.5, stratify=df["y"].values[idx_tmp], random_state=SEED)
tr, va, te = make_split(idx_tr, idx_va, idx_te)
model_b, hist_b = train(tr, va)
res_b = evaluate("Protocolo B", model_b, te, hist_b)
torch.save(model_b.state_dict(), MODELS / "convser_protocolo_b.pt")
res_b

## 3. Resumen

In [ ]:
resumen = pd.DataFrame([res_a, res_b]).set_index("protocolo")
resumen.to_csv(OUT / "resumen_metricas.csv")
(OUT / "resumen_metricas.json").write_text(json.dumps([res_a, res_b], indent=2, default=float))
resumen.style.format({"accuracy": "{:.1%}", "f1_macro": "{:.3f}"})

**Lectura de resultados.** La referencia reporta ~90 % sobre cuatro corpus combinados y con aumentación previa a la división
(lo que infla la métrica). Con solo RAVDESS, sin fuga y —en el protocolo A— con hablantes no vistos, la cifra esperada es menor;
la diferencia entre A y B cuantifica cuánto depende el modelo de conocer al hablante.

## 4. Descargar resultados
En Kaggle, todo lo que está en `/kaggle/working` aparece en la pestaña *Output*.

In [ ]:
import shutil
shutil.make_archive(str(WORK / "resultados_modelo_kaggle"), "zip", WORK, "resultados_modelo")
print(sorted(p.name for p in OUT.iterdir()), sorted(p.name for p in MODELS.iterdir()))